# Germline Whole Exome Sequencing (WES): Variant Discovery

**Author:** Eman Koosehlar

**Version:** *1.1*

**Status:** Ready to run in Google Colab

**Last Updated:** *June 2026*

## Learning Journey

This notebook is the first part of the **Germline Whole Exome Sequencing (WES) Analysis Series**.

The goal of this notebook is to demonstrate how raw sequencing reads (FASTQ files) are processed through quality control, alignment, post-processing, variant calling, and hard filtering to produce a high-confidence VCF file suitable for downstream analysis.

This notebook focuses on **variant discovery** rather than biological interpretation. The resulting filtered VCF will serve as the starting point for variant annotation, prioritization, and clinical interpretation in the subsequent notebooks of this series.


## Overview
This notebook demonstrates a complete germline Whole Exome Sequencing (WES) variant calling workflow using GATK Best Practices.

The pipeline starts from raw FASTQ files and ends with a filtered VCF file containing high-confidence variants.



## Pipeline Overview

<img src="../figures/WES_Pipeline.png" width="800">


<span style="color:red"> This notebook is intended for educational and research purposes.</span>

**Table of contents**<a id='toc0_'></a>    
- [Introduction](#toc1_)
- [Requirements (Tools/Environment/databases)](#toc2_)  
  - [Installation](#toc2_1_)
  - [Environment Settings](#toc2_2_)
  - [Download required files](#toc2_3_)
- [Step 1: FastQC](#toc3_)
- [Step 2: Trimmomatic (Optional)](#toc4_)
- [Step 3: Alignment with BWA-MEM](#toc5_)
- [Step 4: Sort BAM](#toc6_)
- [Step 5: MarkDuplicates](#toc7_)
- [Step 6: Base Quality Score Recalibration (BQSR)](#toc8_)
  - [Apply BQSR](#toc8_1_)
- [Step 7: Variant Calling using HaplotypeCaller](#toc9_)
- [Step 8: Hard Filtering](#toc10_)
- [Summary](#toc11_)
- [References](#toc12_)


<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_'></a>[Introduction](#toc0_)

Whole Exome Sequencing (WES) focuses on sequencing the **protein-coding regions** of the genome (the exome), which represent approximately `1–2%` of the human genome but contain the majority of known disease-causing variants.

The purpose of this workflow is to identify small genetic variants (SNVs and Indels) from exome sequencing data.

This notebook follows the general recommendations of the `GATK Best Practices` pipeline for germline variant discovery.

---


## <a id='toc2_'></a>[Requirements (Environment/Tools/databaces)](#toc0_)



The following tools are required:

| Tool | Purpose |
|--------|---------|
| FastQC | Read quality assessment |
| Multiqc | Create Interactive report |
| Trimmomatic | Adapter and quality trimming |
| BWA-MEM | Alignment |
| Samtools | BAM processing |
| GATK | Variant calling |
| Picard | Duplicate marking |

 **Project Directories**

To keep the workflow organized and reproducible, all files generated during the analysis are stored in dedicated project directories.

```text
WES_Project/
│
│
├── Reference/ (✅Current Notebook)
│   ├── Reference_Genome hg38
│   └── Gatk_Known_Sites
│       ├── dbsnp138
│       ├── hapmap_3.3
│       ├── 1000G_omni2.5.hg38
│       ├── 1000G_phase1.snps.high_confidence.hg38
│       ├── Homo_sapiens_assembly38.known_indels
│       ├── Mills_and_1000G_gold_standard.indels.hg38
│       ├── wgs_calling_regions.hg38.interval_list
│       └── hg38.scattered_calling_intervals/  
│ 
│ 
│ 
├── Alignment/ (✅Current Notebook)
│   └── Mapped Reads Files
│
│
├── Germline_Variant_Calls/ (✅Current Notebook)
│   └── Raw VCF files
│
│
├── VCF/ (✅Current Notebook)
│   ├── Filtered VCF files
│   └── Final VCF used for annotation
│
│
├── Annotation/ (🚧 In 02_Germline_Variant_Annotation Notebook)
│   ├── Annotated VCF files
│   └── Annotation reports
│
│
├── Prioritization/ (🚧 In 02_Germline_Variant_Annotation Notebook)
│   └── Filtered annotated VCF files ready for downstream interpretation
│
│
└── Results/ (🚧 In 03_Germline_Variant_Interpretation Notebook)
    └── Summary and final outputs
```

### Directory Description

* **Reference/** – Stores the reference genome and known variant databases required by GATK and VEP.

* **Germline_Variant_Calls/** - Stores VCF files generated after variant calling

* **VCF/** – Stores VCF files generated after hard filtering.

* **Alingment/** – Stores maped reads file after alingning reads to the reference genome



### <a id='toc2_1_'></a>[Installation](#toc0_)

In [ ]:
# Java
!apt install openjdk-17-jdk -y

# FastQC
!apt install fastqc -y

# Multiqc
!apt-get install multiqc

# Samtools
!apt install samtools -y

# Picard
!apt-get install picard

# BWA
!apt install bwa -y

# Trimmomatic
!apt install trimmomatic -y

# Download GATK
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

!unzip -q gatk-4.6.2.0.zip

- Check the tools with the fullowing commands.

In [ ]:
!fastqc --help

In [ ]:
!samtools --help

In [ ]:
!bwa

In [ ]:
!gatk --help

In [ ]:
!java -jar /content/picard.jar -h

----
- **Optional installation for samtools (if the above section had error use this)**


!wget https://github.com/samtools/samtools/releases/download/1.7/samtools-1.7.tar.bz2


!tar --bzip2 -xvf samtools-1.7.tar.bz2

cd samtools-1.7/

!make

!make install

cd /content

!rm samtools-1.7.tar.bz2

`use each command in a separate cells`

- **Picard installation (optional)**


!wget https://github.com/broadinstitute/picard/releases/download/2.18.14/picard.jar




----

### <a id='toc2_2_'></a>[Environment settings](#toc0_)

 Environment setting and creating required directories for the follwing steps

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Create directory for alingment section
!mkdir WES_Analysis

!mkdir WES_Analysis/Alignment

# Creating dir for genome file
!mkdir -p WES_Analysis/Reference/Reference_Genome

# Creating dir for Known site Databases
!mkdir WES_Analysis/Reference/Gatk_Known_Sites

# Create dir for VCF files
!mkdir WES_Analysis/Germline_Variant_Calls WES_Analysis/VCF



In [ ]:
#Environment settings

CHRS = 'chr6_and_chr17'
GATK_REGIONS='-L chr6 -L chr17'
Known_Sites = '/content/WES_Analysis/Reference/Gatk_Known_Sites'
Germline_Calls = '/content/WES_Analysis/Germline_Variant_Calls'
VCF = '/content/WES_Analysis/VCF'
Reference_Genome = '/content/WES_Analysis/Reference/Reference_Genome'
Alignment = '/content/WES_Analysis/Alignment'


### <a id='toc2_3_'></a>[Download required files](#toc0_)

1. **Raw Fastq files**

In [ ]:
# Download and extract fastq files
!wget https://genomedata.org/pmbio-workshop/fastqs/chr6_and_chr17/Exome_Norm.tar
!tar -xvf Exome_Norm.tar


- *Some exploration on raw data (optional)*


In [ ]:
# look at the header of the file (same proccess for the R2 file)
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | head

# what do R1 and R2 refer to? What is the length of each read?
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | head -n 2 | tail -n 1 | wc

# how many lines are there in the Exome_Tumor file
!zcat Exome_Norm/Exome_Norm_R1.fastq.gz | wc -l # There are: 33,326,620

# how many paired reads or fragments are there then?
!expr 25466068 / 4 # There are: 8,331,655 paired end reads

# how many total bases of data are in the Exome Tumor data set?
!echo "6366517 * (101 * 2)" | bc # There are: 1,682,994,310 bases of data

# how many total bases when expressed as "gigabases" (specify 2 decimal points using `scale`)
!echo "scale=2; (6366517 * (101 * 2))/1000000000" | bc # There are: 1.68 Gbp of data


2. **Downloding Reference Genome**

In [ ]:
# Downloading refrence genome
!wget -P {Reference_Genome}  http://genomedata.org/pmbio-workshop/references/genome/$CHRS/ref_genome.tar

In [ ]:
# Extracting ref genome
!tar -xvf {Reference_Genome}/ref_genome.tar

In [ ]:
# Removing unused tar file
rm {Reference_Genome}/ref_genome.tar

In [ ]:
# Changing directory to /content
cd /content/

In [ ]:
# move all extracted ref files to its directories
!mv ref_genome.* {Reference_Genome}

In [ ]:
# Unzip ref genome file
!gunzip {Reference_Genome}/ref_genome.fa.gz

- *Checking ref genome file contents:*

In [ ]:
# Inspecting header of the ref genome file
!head {Reference_Genome}/ref_genome.fa

In [ ]:
# How long are to two chromosomes combined (in bases and Mbp)? Use grep to skip the header lines for each chromosome.
!grep -v ">" {Reference_Genome}/ref_genome.fa | wc


# How long does that command take to run?
!time grep -v ">" {Reference_Genome}/ref_genome.fa | wc

# View 10 lines from approximately the middle of this file
!head -n 2500000 {Reference_Genome}/ref_genome.fa | tail


In [ ]:
# What is the count of each base in the entire reference genome file (skipping the header lines for each sequence)?


from collections import Counter

base_counts = Counter()

with open('/content/WES_Analysis/Reference/Reference_Genome/ref_genome.fa') as f:
    for line in f:
        if line.startswith('>'):
            continue
        base_counts.update(line.strip())

# Print sorted base counts
for base in sorted(base_counts):
    print(f"{base} {base_counts[base]}")




In [ ]:
# How many time a pattern like 'GAATTC' occurs this genome?

import re

pattern = re.compile(r'(?=GAATTC)')  # Lookahead to allow overlapping matches
chunks = []

# Read and collect all lines except headers
with open('/content/WES_Analysis/Reference/Reference_Genome/ref_genome.fa') as f:
    for line in f:
        if not line.startswith('>'):
            chunks.append(line.strip())

sequence = ''.join(chunks)  # Efficient one-time join
count = len(pattern.findall(sequence))

print(f"The pattern 'GAATTC' occurs {count} times in the genome.")

# The pattern 'GAATTC' occurs 71525 times in the genome.

3. **Downloading Known sites databases**

In [ ]:
cd {Known_Sites}

In [ ]:
!gsutil -o "GSUtil:check_hashes=never" cp gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.dbsnp138.vcf .

In [ ]:
!gzip {Known_Sites}/Homo_sapiens_assembly38.dbsnp138.vcf

In [ ]:
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/hapmap_3.3.hg38.vcf.gz .

In [ ]:
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/1000G_omni2.5.hg38.vcf.gz .
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz .

In [ ]:
# Indel calibration call sets - dbsnp, Mills
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.known_indels.vcf.gz .
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz .

In [ ]:
# Interval lists that can be used to parallelize certain GATK tasks
!gsutil cp gs://genomics-public-data/resources/broad/hg38/v0/wgs_calling_regions.hg38.interval_list .
!gsutil cp -r gs://genomics-public-data/resources/broad/hg38/v0/scattered_calling_intervals/ .

In [ ]:
!gunzip {Known_Sites}/Homo_sapiens_assembly38.dbsnp138.vcf.gz

**<span style = 'color:blue'> Outputs generated during this notebook can be stored in:</span>**

- Google Drive (*recommended*)
- Local Colab runtime (*temporary*)

Important intermediate files such as reference genomes,
`BAM` files, and `filtered VCF` files can be reused in
downstream annotation and variant prioritization notebooks.

---

## <a id='toc3_'></a>[Step 1: FastQC](#toc0_)

**Purpose**

FastQC evaluates the quality of raw sequencing reads before alignment.

Important metrics include:

- Per-base sequence quality
- Adapter contamination
- GC content
- Sequence duplication levels

**Why is this important?**

Poor quality reads may lead to:

- Alignment errors
- False-positive variants
- Reduced variant calling accuracy

**Input files:**

- R1 FASTQ
- R2 FASTQ


**Expected output:**

- Quality Control's HTML Reports 

In [ ]:
# Change to the working directory
cd /content/

In [ ]:
# Run fastqc
!fastqc Exome_Norm/Exome_Norm*.fastq.gz -o Exome_Norm/FASTQC/

Check the results by download and view them in your browser. 

- **QC Interpretation**

For WES data, we typically expect:

✓ Per-base quality scores > Q30

✓ Low adapter contamination

✓ Consistent GC distribution

If the reports are acceptable, trimming may not be necessary.

## <a id='toc4_'></a>[Step 2: Trimmomatic (Optional)](#toc0_)

**Purpose**

Trimmomatic removes:

- Adapter sequences
- Low-quality bases
- Poor-quality reads

**Is it required?**

Not always.

In this project, FastQC showed high-quality reads with minimal adapter contamination, therefore trimming was not required.

However, this step is included for completeness because many datasets benefit from trimming before alignment.

In [ ]:
!trimmomatic PE \
Exome_Norm/Exome_Norm_R1.fastq.gz Exome_Norm/Exome_Norm_R2.fastq.gz \
R1_paired.fastq.gz R1_unpaired.fastq.gz \
R2_paired.fastq.gz R2_unpaired.fastq.gz \
SLIDINGWINDOW:4:20 MINLEN:50

## <a id='toc5_'></a>[Step 3: Alignment using BWA-MEM](#toc0_)

**Purpose**

Alignment maps sequencing reads to the reference genome.

Input:

- FASTQ files

Output:

- SAM/BAM file

**Why?**

Variant callers require genomic coordinates
for each sequencing read.

In [ ]:
# Chenge directory to ref genome
cd {Reference_Genome}

1. **Creating refrence's index files**

**What does this command do?**

This command prepares the reference genome for alignment by creating index files.

**Why is it needed?**

BWA cannot align sequencing reads directly to a raw FASTA file. It first needs to build an index that allows it to quickly search and locate matching regions in the genome.

**Input**
- Reference genome FASTA file (ref_genome.fa)

**Output**

- Several index files are generated, including:
1. ref_genome.fa.amb
2. ref_genome.fa.ann
3. ref_genome.fa.bwt
4. ref_genome.fa.pac
5. ref_genome.fa.sa

In [ ]:
# Creating index file from ref genome

!bwa index {Reference_Genome}/ref_genome.fa # Produce ~670 Mb ref files

2. **Runing alignmet to the reference genome**

In [ ]:
# Run alignmet step with bwa mem
# takes 5.4 min

!bwa mem -t 8 -Y -R "@RG\tID:Exome_Norm\tPL:ILLUMINA\tPU:C1TD1ACXX-CGATGT.7\tLB:exome_norm_lib1\tSM:HCC1395BL_DNA" {Reference_Genome}/ref_genome.fa /content/Exome_Norm/Exome_Norm_R1.fastq.gz /content/Exome_Norm/Exome_Norm_R2.fastq.gz > {Alignment}/Exome_Norm.sam

3. **Checking alignmet results with samtools**

Alingment step produces `Exome_Norm.sam` in a totla volume of 4.3 Gb including maped reads to reference genome and statistical contents. 

In [ ]:
!samtools flagstat {Alignment}/Exome_Norm.sam

## <a id='toc6_'></a>[Step 4: Sort BAM](#toc0_)

**Purpose**

BAM files must be coordinate sorted before
downstream GATK processing.

Sorting organizes reads according to
their genomic positions.

In [ ]:
# Create bam (a binary/machine readable) file 
!samtools view -@ 8 -h -b -o Exome_Norm.bam {Alignment}/Exome_Norm.sam

In [ ]:
# Create bam (a binary/machine readable) file 
!samtools view -@ 8 -h -b -o Exome_Norm.bam {Alignment}/Exome_Norm.sam

## <a id='toc7_'></a> [Step 5: MarkDuplicates](#toc0_)

**Purpose**

PCR amplification during library preparation
can produce duplicate reads.

These duplicates do not represent independent
DNA observations and may bias variant calling.

MarkDuplicates identifies these reads and marks them.

In [ ]:
# Run 
!java -jar picard.jar MarkDuplicates I={Alignment}/Exome_Norm_namesorted.bam O={Alignment}/Exome_Norm_namesorted_mrkdup.bam ASSUME_SORT_ORDER=queryname METRICS_FILE=Exome_Norm_mrkdup_metrics.txt QUIET=true COMPRESSION_LEVEL=0 VALIDATION_STRINGENCY=LENIENT

In [ ]:
# Position sort bam file
!java -jar picard.jar SortSam \
I={Alignment}/Exome_Norm_namesorted_mrkdup.bam O={Alignment}/Exome_Norm_sorted_mrkdup.bam SO=coordinate

In [ ]:
# View Marked duplicate metrics
!cat /content/Exome_Norm_mrkdup_metrics.txt

In [ ]:
# Statistic info of dups reads
!samtools flagstat {Alignment}/Exome_Norm_sorted_mrkdup.bam

In [ ]:
#create bam index
!java -jar picard.jar BuildBamIndex I={Alignment}/Exome_Norm_sorted_mrkdup.bam

In [ ]:
# Optional way to create bam index by using gatk:
# !gatk BuildBamIndex -I {Alignment}/Exome_Norm_sorted_mrkdup.bam

## <a id='toc8_'></a>[Step 6: Base Quality Score Recalibration (BQSR)](#toc0_)

**Purpose**

Sequencing machines introduce systematic
errors in base quality scores.

BQSR models these errors using known variant sites
and recalibrates base quality values.

**Why?**

Accurate quality scores improve variant calling accuracy.

This step needed information of known human variants we already download it before in ([here](#toc2_3_))



1. **Biulding Index Files for the Known Variants**



**What does this command do?**

This command creates an index file for the dbSNP VCF file.

**Why is it needed?**

GATK tools need indexed files so they can quickly locate variants in specific genomic regions without reading the entire file.

**Input**

- dbSNP VCF file

**Output**
- An index file (`.idx`) generated alongside the VCF file.


**Why is dbSNP/1000G/hapmap used?**

These databases contain known human variants and is used during Base Quality Score Recalibration (BQSR) to distinguish true biological variants from sequencing errors.

In [ ]:
# SNP calibration call sets - dbsnp, hapmap, omni, and 1000G
# Runtime: ~ 4min each
!gatk IndexFeatureFile -I {Known_Sites}/Homo_sapiens_assembly38.dbsnp138.vcf

!gatk --java-options '-Xmx24g' IndexFeatureFile -I {Known_Sites}/hapmap_3.3.hg38.vcf.gz

!gatk --java-options '-Xmx24g' IndexFeatureFile -I {Known_Sites}/1000G_omni2.5.hg38.vcf.gz

!gatk --java-options '-Xmx24g' IndexFeatureFile -I {Known_Sites}/1000G_phase1.snps.high_confidence.hg38.vcf.gz

#Indel calibration call sets - dbsnp, Mills
!gatk --java-options '-Xmx24g' IndexFeatureFile -I {Known_Sites}/Homo_sapiens_assembly38.known_indels.vcf.gz

!gatk --java-options '-Xmx24g' IndexFeatureFile -I {Known_Sites}/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz



2. **Runnign BQSR**

In [ ]:
# Make sure to have downloaded all of the requered data for known sites and run the previous step
!gatk --java-options '-Xmx60g' BaseRecalibrator -R {Reference_Genome}/ref_genome.fa -I {Alignment}/Exome_Norm_sorted_mrkdup.bam -O {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.table --known-sites /content/ref/gatk_Known_site/Homo_sapiens_assembly38.dbsnp138.vcf --known-sites /content/ref/gatk_Known_site/Homo_sapiens_assembly38.known_indels.vcf.gz --known-sites /content/ref/gatk_Known_site/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --preserve-qscores-less-than 6 --disable-bam-index-caching $GATK_REGIONS

3. **Checking statistical information**

In [ ]:
# Viewing the table result
!cat {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.table

In [ ]:
# Viewing statistical information with samtools
!samtools flagstat {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.bam

4. **Put Together BQSR Data**

In [ ]:
!java -jar picard.jar CollectAlignmentSummaryMetrics R={Reference_Genome}/ref_genome.fa I={Alignment}/Exome_Norm_sorted_mrkdup_bqsr.bam O=metrics.txt

In [ ]:
# Viewing the text file
!cat metrics.txt

### <a id='toc8_1_'></a>[ApplyBQSR](#toc0_)


**What does this step do?**

In the previous step, GATK identified systematic errors in the base quality scores assigned by the sequencing machine.

`ApplyBQSR` uses that information to correct the quality scores in the BAM file.

**Why is this important?**

Variant callers rely heavily on base quality scores when deciding whether a nucleotide difference is a true variant or a sequencing error.

More accurate quality scores lead to more reliable variant calls.

**Input**

- Deduplicated `BAM` file
- Recalibration `table` generated by *BaseRecalibrator*

**Output**

* Recalibrated `BAM` file

**Simple analogy**

Think of BaseRecalibrator as creating a list of corrections for an exam grading system.

ApplyBQSR is the step where those corrections are actually applied to the students' scores.

After this step, the recalibrated BAM file is used for variant calling with HaplotypeCaller.


**1. Running ApplyBQSR**

You can do this step with this command:

`!gatk --java-options '-Xmx60g' -Xmx60g' ApplyBQSR -R {Reference_Genome}/ref_genome.fa -I {Alignment}/Exome_Norm_sorted_mrkdup.bam -O {Alignment}/Exome_Norm_bqsr.bam --bqsr-recal-file {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.table --preserve-qscores-less-than 6 --static-quantized-quals 10 --static-quantized-quals 20 --static-quantized-quals 30`

or optionally run the below Python code to run it. 

In [ ]:
import os
import subprocess

def run_base_recalibrator(ref_genome, bam_file, output_table, known_sites, regions=""):
    """
    Run the GATK BaseRecalibrator step.

    Parameters:
    - ref_genome: Path to the reference genome file.
    - bam_file: Path to the sorted and marked duplicates BAM file.
    - output_table: Path to the output table file where recalibration results will be stored.
    - known_sites: List of known sites VCF file paths.
    - regions: Optional parameter specifying genomic regions (e.g., --intervals for specific chromosomes).

    Returns:
    - Command output or error message.
    """

    # Construct known-sites arguments
    known_sites_args = ' '.join([f"--known-sites {vcf}" for vcf in known_sites])

    # Build the BaseRecalibrator command
    command = f"gatk --java-options '-Xmx60g' BaseRecalibrator " \
              f"-R {ref_genome} " \
              f"-I {bam_file} " \
              f"-O {output_table} " \
              f"{known_sites_args} " \
              f"--preserve-qscores-less-than 6 " \
              f"--disable-bam-index-caching {regions}"

    # Run the command
    print(f"Running BaseRecalibrator with command: {command}")
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(result.stdout.decode())  # Print stdout (log)
        return result.stdout.decode()
    except subprocess.CalledProcessError as e:
        print(f"Error running BaseRecalibrator: {e.stderr.decode()}")
        return e.stderr.decode()


def run_apply_bqsr(ref_genome, bam_file, recal_table, output_bam, regions=""):
    """
    Run the GATK ApplyBQSR step.

    Parameters:
    - ref_genome: Path to the reference genome file.
    - bam_file: Path to the recalibrated BAM file.
    - recal_table: Path to the recalibration table file.
    - output_bam: Path to the output BAM file after applying BQSR.
    - regions: Optional parameter specifying genomic regions (e.g., --intervals for specific chromosomes).

    Returns:
    - Command output or error message.
    """

    # Build the ApplyBQSR command
    command = f"gatk --java-options '-Xmx60g' ApplyBQSR " \
              f"-R {ref_genome} " \
              f"-I {bam_file} " \
              f"-O {output_bam} " \
              f"--bqsr-recal-file {recal_table} " \
              f"--preserve-qscores-less-than 6 " \
              f"--static-quantized-quals 10 " \
              f"--static-quantized-quals 20 " \
              f"--static-quantized-quals 30 " \
              f"{regions}"

    # Run the command
    print(f"Running ApplyBQSR with command: {command}")
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(result.stdout.decode())  # Print stdout (log)
        return result.stdout.decode()
    except subprocess.CalledProcessError as e:
        print(f"Error running ApplyBQSR: {e.stderr.decode()}")
        return e.stderr.decode()

# Example usage of the functions
def pipeline_example():
    ref_genome = "/content/WES_Analysis/Reference/Reference_Genome/ref_genome.fa"
    bam_file = "/content/WES_Analysis/Alignment/Exome_Norm_sorted_mrkdup_bqsr.bam"
    output_table = "/content/WES_Analysis/Alignment/Exome_Norm_sorted_mrkdup_bqsr.table"
    output_bam = "/content/WES_Analysis/Alignment/Exome_Norm_bqsr.bam"

    # List of known sites VCF files
    known_sites = [
        "/path/to/dbsnp138.vcf",
        "/path/to/known_indels.vcf.gz",
        "/path/to/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz"
    ]

    # Run BaseRecalibrator
    #run_base_recalibrator(ref_genome, bam_file, output_table, known_sites)

    # Run ApplyBQSR
    run_apply_bqsr(ref_genome, bam_file, output_table, output_bam)

# Execute pipeline
pipeline_example()


**2. Rechecking The Quality of Reads**

Using `fastqc` to check the quality of reads after recalibration step:

In [ ]:
#post alignmet QC
!fastqc {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.bam


**3. Making Interactive Report**

Creating `.txt` from bam file stats info. then Produce `.html` report file with `multiqc`. It is using all txt in the directory to create the report. You should open it in your browser after downloading it. 

## <a id='toc9_'></a>[Step 7: Variant Calling using HaplotypeCaller](#toc0_)

**Purpose**

Identify SNPs and small insertions/deletions (Indels).

HaplotypeCaller performs local reassembly of reads,
which improves variant detection compared to
simple pileup-based approaches.

**Output:**

- VCF containing candidate variants

In [ ]:
#Variant Calling step with Haplotypecaller (Takes ~12 min)
!gatk --java-options '-Xmx60g' HaplotypeCaller -R {Reference_Genome}/ref_genome.fa -I {Alignment}/Exome_Norm_sorted_mrkdup_bqsr.bam -O {Germline_Calls}/Exome_Norm_calls.vcf --bam-output {Germline_Calls}/Exome_Norm_out.bam $GATK_REGIONS


## <a id='toc10_'></a>[Step 8: Hard Filtering](#toc0_)


**Purpose**

Raw variant calls contain false positives.

Hard filtering removes low-confidence variants
based on quality metrics.

This approach is recommended when Variant Quality
Score Recalibration (VQSR) is not feasible,
which is often the case for single-sample WES analyses.

In [ ]:
Germline_Variant_Calls = {Germline_Calls}
VCF = '/content/VCF'

In [ ]:
#selectvariants

!gatk --java-options '-Xmx60g' SelectVariants -R {Reference_Genome}/ref_genome.fa -V {Germline_Calls}/Exome_Norm_calls.vcf -select-type SNP -O {VCF}/Exome_Norm_calls.snps.vcf
!gatk --java-options '-Xmx60g' SelectVariants -R {Reference_Genome}/ref_genome.fa -V {Germline_Calls}/Exome_Norm_calls.vcf -select-type INDEL -O {VCF}/Exome_Norm_calls.indels.vcf

Check the header or number of snps or indels by following codes. 

In [ ]:
!head {VCF}/Exome_Norm_calls.snps.vcf


In [ ]:
!grep -v "^#" {VCF}/Exome_Norm_calls.snps.vcf | wc -l

In [ ]:
!grep -v "^#" {VCF}/Exome_Norm_calls.indels.vcf | wc -l

In [ ]:
# SNP filtering step
!gatk --java-options '-Xmx64g' VariantFiltration -R {Reference_Genome}/ref_genome.fa -V {Germline_Calls}/Exome_Norm_calls.snps.vcf --filter-expression "QD < 2.0" --filter-name "QD_lt_2" --filter-expression "FS > 60.0" --filter-name "FS_gt_60" --filter-expression "MQ < 40.0" --filter-name "MQ_lt_40" --filter-expression "MQRankSum < -12.5" --filter-name "MQRS_lt_n12.5" --filter-expression "ReadPosRankSum < -8.0" --filter-name "RPRS_lt_n8" --filter-expression "SOR > 3.0" --filter-name "SOR_gt_3" -O {VCF}/Exome_Norm_calls.snps.filtered.vcf

# Indel filtering step
!gatk --java-options '-Xmx64g' VariantFiltration -R {Reference_Genome}/ref_genome.fa -V {Germline_Calls}/Exome_Norm_calls.indels.vcf --filter-expression "QD < 2.0" --filter-name "QD_lt_2" --filter-expression "FS > 200.0" --filter-name "FS_gt_200" --filter-expression "ReadPosRankSum < -20.0" --filter-name "RPRS_lt_n20" --filter-expression "SOR > 10.0" --filter-name "SOR_gt_10" -O {VCF}/Exome_Norm_calls.indels.filtered.vcf

Checking how many variant is passed by this step. compare befor and after filteration step. 

Check Variant Filter Status by this command.

`!grep -v "^#" /content/WES_Analysis/VCF/Exome_Norm_calls.snps.filtered.vcf | cut -f7 | sort | uniq -c`


What does this command do?

This command summarizes the filter status of all variants in the VCF file.

Step by step

- `grep -v "^#"` → removes header lines from the VCF file.
- `cut -f7` → extracts the FILTER column (column 7).
- `sort` → sorts the filter values.
- `uniq -c` → counts how many times each filter status appears.

Why is this useful?

It provides a quick overview of how many variants:

Passed all filters (PASS)
Failed specific filtering criteria (e.g., QD2, FS60, MQ40)
Example Output
12543 PASS
234 QD2
87 FS60
15 MQ40
Interpretation

In this example:

- 12,543 variants passed all quality filters.
- 234 variants failed the Quality by Depth (QD) filter.
- 87 variants failed the Fisher Strand Bias (FS) filter.
- 15 variants failed the Mapping Quality (MQ) filter.

This helps assess the impact of the filtering step on the variant dataset.

In [ ]:

!grep -v "^#" {VCF}/Exome_Norm_calls.snps.filtered.vcf | grep "PASS" | wc -l

You can also check these information with the below py code (just for snp as an example).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Set your VCF file path
vcf_file = "/content/WES_Analysis/VCF/Exome_Norm_calls.snps.filtered.vcf"

# Read the VCF and skip header lines
filter_counts = {}

with open(vcf_file, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        filters = fields[6].split(";")  # Split multiple filters
        for flt in filters:
            filter_counts[flt] = filter_counts.get(flt, 0) + 1

# Convert to DataFrame
df = pd.DataFrame(list(filter_counts.items()), columns=["Filter", "Count"])
df = df.sort_values(by="Count", ascending=False)

# Show the DataFrame
print(df)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(df["Filter"], df["Count"], color="skyblue")
plt.xlabel("Number of Variants")
plt.title("VCF FILTER Field Summary")
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


- Merging filtered SNP and INDEL vcfs back together

In [ ]:
!gatk --java-options '-Xmx60g' MergeVcfs -I {VCF}/Exome_Norm_calls.snps.filtered.vcf \
-I {VCF}/Exome_Norm_calls.indels.filtered.vcf \
-O {VCF}/Exome_Norm_calls.filtered.vcf

In [ ]:
!grep -v "^#" {VCF}/Exome_Norm_calls.filtered.vcf | cut -f7 | sort | uniq -c

In [ ]:
# Checking how many vars passed
!grep -v "^#" {VCF}/Exome_Norm_calls.filtered.vcf | grep "PASS"  | wc -l

- Extracting only PASS variants

In [ ]:
!gatk --java-options '-Xmx60g' SelectVariants -R {Reference_Genome}/ref_genome.fa \
-V {VCF}/Exome_Norm_calls.filtered.vcf \
-O {VCF}/Exome_Norm_calls.filtered.PASS.vcf --exclude-filtered

- Saving the final vcf file to the Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Create project directory
!mkdir /content/gdrive/MyDrive/WES_Analysis/VCF

In [ ]:
!cp {VCF}/Exome_Norm_calls.filtered.PASS.vcf \
/content/gdrive/MyDrive/WES_Analysis/VCF/

!cp {VCF}/Exome_Norm_calls.filtered.PASS.vcf.idx \
/content/gdrive/MyDrive/WES_Analysis/VCF/

## <a id='toc11_'></a>[Summary](#toc0_)

In this notebook, we implemented a complete germline Whole Exome Sequencing (WES) variant calling workflow following the general principles of GATK Best Practices.


**<span style = 'color:purple'> 
The analysis included:</span>**

1. Quality assessment of raw sequencing reads using `FastQC`.
2. Optional read trimming with `Trimmomatic`.
3. Alignment of sequencing reads to the reference genome using `BWA-MEM`.
4. BAM file sorting and processing with `Samtools` and `GATK`.
5. Identification and marking of PCR duplicates.
6. Base Quality Score Recalibration (BQSR).
7. Germline variant calling using `GATK HaplotypeCaller`.
8. Hard filtering of SNPs and Indels to obtain *high-confidence variants*.

The final output of this workflow is a *filtered VCF file* containing candidate germline variants that are ready for downstream annotation and interpretation.

**<span style = 'color:purple'> Next Steps </span>**

This notebook focuses on variant discovery and filtering only.

Downstream analyses may include:

* Variant annotation using VEP or ANNOVAR
* Population frequency filtering
* Functional consequence assessment
* Phenotype-driven prioritization
* ACMG-guided variant interpretation



<span style = 'color:red'> **Important Note:** Variant filtering is not equivalent to clinical interpretation.</span>

Additional evidence such as population frequency, functional studies, segregation analysis, phenotype matching, and published literature should be considered before assigning clinical significance to a variant.


## <a id='toc12_'></a>[References](#toc0_)


1. Van der Auwera GA, O'Connor BD. *Genomics in the Cloud: Using Docker, GATK, and WDL in Terra*. O'Reilly Media, 2020.

2. GATK Best Practices Documentation (Broad Institute).

3. Li H. (2013). Aligning sequence reads, clone sequences and assembly contigs with BWA-MEM.

4. Andrews S. FastQC: A Quality Control Tool for High Throughput Sequence Data.

5. Bolger AM, Lohse M, Usadel B. (2014). Trimmomatic: A flexible read trimming tool for Illumina sequence data.

6. Sherry ST et al. dbSNP: The NCBI database of genetic variation.

7. Genome Reference Consortium. Human Genome Assembly GRCh38.
